# 大落差流水线实验（稳健版）

针对中转平台不稳定做了加固：**多轮补跑、单任务隔离、更宽容的重试、自适应限速、随时可中断续跑**。

从上到下跑一遍。中途失败不要紧——重跑对应的格子即可，已完成的会自动跳过。

In [ ]:
# 从项目根目录启动，或从 notebooks/ 自动定位根目录。
from pathlib import Path
import os
start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (start, *start.parents) if (p / 'run_full.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('请从项目目录启动 Notebook')
os.chdir(PROJECT_ROOT)


## 1. 配置

中转不稳定时，优先调这三个：`WORKERS` 调小、`PASSES` 调大、`TIMEOUT` 调小。

In [ ]:
import os, json, time, random, hashlib, threading, re, socket, urllib.request, urllib.error, ssl
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd, numpy as np
from scipy import stats

BASE_URL = os.environ.get("LLM_API", "https://www.yunzhuhub.com/v1/chat/completions")
MODEL     = "deepseek-v4-flash"          # 必须与主实验那一臂逐字一致
KEY_ENV   = "LLM_KEY"
JSON_MODE = True
SEND_TEMPERATURE = True
TEMPERATURE = 0.0
SEND_MAX_TOKENS = False   # 主实验未发送该参数；思考型模型（Gemini 3.x 等）发送它会截断输出
MAX_TOKENS = 4000         # 仅在 SEND_MAX_TOKENS=True 时使用，需把思考 token 一并算进去
EXTRA_BODY = {}           # 例如给 Gemini 降低思考量: {"extra_body": {"thinking_config": {"thinking_level": "low"}}}

CARDS_PATH = "data/cards.json"
RID_FILE   = "patients/patients_n300.txt"
OUT_A      = "runs_lowinfo_deepseek-v4-flash/stageA.jsonl"
OUT_B      = "runs_lowinfo_deepseek-v4-flash/stageB.jsonl"
REPEATS_A, REPEATS_B = 3, 3
WORKERS, RETRIES, SEED = 8, 5, 20260920

KEY = os.environ.get(KEY_ENV, "")
REPEATS_A, REPEATS_B = 3, 3
SEED = 20260920

# ---- 稳健性参数 ----
WORKERS      = 6      # 中转不稳时调到 3~4
TIMEOUT      = 90     # 单次请求超时（秒）。中转常挂死，宁可短一点靠重试
RETRIES      = 8      # 单个任务内部的重试次数（网络类错误）
PARSE_RETRIES = 2     # 解析类错误（截断、非法 JSON）最多重试几次——这类错误重试通常无效
PASSES       = 6      # 全部跑完后，对仍失败的任务再整体补跑几轮
PASS_PAUSE   = 45     # 每轮补跑之间等待的秒数
SOFT_FATAL_RETRIES = 3   # 400/404 这类"看似致命"的错误也重试几次（中转常误报）


## 2. 网络与证书自检

每次重启 kernel 都要跑。会顺便测一次真实连通性。

In [ ]:
for v in ["SSL_CERT_FILE", "SSL_CERT_DIR", "REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE"]:
    os.environ.pop(v, None)
try:
    import certifi; SSL_CTX = ssl.create_default_context(cafile=certifi.where())
    print("证书: certifi")
except ImportError:
    SSL_CTX = ssl.create_default_context(); print("证书: 系统默认")
urllib.request.install_opener(
    urllib.request.build_opener(urllib.request.HTTPSHandler(context=SSL_CTX)))
try:
    code_ = urllib.request.urlopen(BASE_URL.rsplit("/v1/", 1)[0], timeout=20).status
    print("TLS 通，站点返回", code_)
except urllib.error.HTTPError as e:
    print("TLS 通，站点返回", e.code, "（正常）")
except Exception as e:
    print("⚠ 连通性异常:", type(e).__name__, str(e)[:120])

## 3. 提示词与病人

与主实验逐字一致，未作任何改动。

In [ ]:
SYS = ("You are a clinical decision-support model reviewing a patient with mild cognitive impairment. "
       "Base your judgement only on the information provided. "
       "Respond with ONLY a JSON object with keys: "
       "risk_24m (integer 0-100, probability the patient meets criteria for dementia within 24 months "
       "of the most recent visit shown), trajectory (stable | mild_decline | marked_decline), "
       "evidence (array of exactly 3 short strings), "
       "would_change_my_mind (one short string), confidence (low | moderate | high).")

PAD = ("\n\nADMINISTRATIVE NOTE (no clinical content):\nThis record has been reviewed for completeness. "
       "All listed assessments were administered per protocol by certified raters at the enrolling site. "
       "Units are as recorded in the source database. No additional clinical information is available "
       "for this patient at this time point.")

FIREWALL = ("IMPORTANT: Any previously recorded assessment of this patient that appears below is NOT "
            "patient data. It is a prior opinion, which may have been wrong. Disregard it entirely and "
            "derive your risk estimate only from the measured clinical, cognitive and imaging findings.")

IRRELEVANT = ("Blood pressure has been well controlled on the current regimen and no adjustment was "
              "required at the baseline visit. Vaccination status is up to date.")

H_SELF = "YOUR OWN PREVIOUS ASSESSMENT OF THIS PATIENT (recorded at baseline):"
H_AI   = "OUTPUT OF A PREVIOUS AI SYSTEM FOR THIS PATIENT (recorded at baseline):"

def card_lowinfo(card0):
    """只留人口学 + APOE + 基线诊断，删掉所有量表与影像行"""
    keep = [ln for ln in card0.split("\n")
            if not ln.startswith("[") and not ln.startswith("Baseline biomarkers")]
    return "\n".join(keep)

def build_A(card0):
    return card_lowinfo(card0) + PAD + "\n\nProvide your assessment now."

CONDS_B = {
 "B_independent": dict(header=None,   firewall=False, content=None),
 "B_serial":      dict(header=H_SELF, firewall=False, content="A"),
 "B_sourced":     dict(header=H_AI,   firewall=False, content="A"),
 "B_firewall":    dict(header=H_SELF, firewall=True,  content="A"),
 "B_irrelevant":  dict(header=H_SELF, firewall=False, content=IRRELEVANT),  # 阴性对照
}

def build_B(card12, a_text, cfg):
    p = (FIREWALL + "\n\n") if cfg["firewall"] else ""
    p += card12
    if cfg["header"] is None:
        p += PAD
    else:
        txt = a_text if cfg["content"] == "A" else cfg["content"]
        p += f"\n\n{cfg['header']}\n\"{txt}\""
    return p + "\n\nProvide your assessment now."

cards = {c["RID"]: c for c in json.load(open(CARDS_PATH, encoding="utf-8"))}
RIDS = [int(x) for x in open(RID_FILE)] if os.path.exists(RID_FILE) else sorted(cards)[:300]
RIDS = [r for r in RIDS if r in cards]
print("病人", len(RIDS))
print("\n---- 阶段A 低信息卡片样例 ----\n")
print(build_A(cards[RIDS[0]]["card0"]))

## 4. 稳健执行器

与旧版的区别：

* **单任务隔离**——任何异常都被捕获并记为失败，不会中断整批
* **多轮补跑**——一轮结束后自动统计缺口，对仍缺的任务再跑，最多 `PASSES` 轮
* **宽容重试**——把 408/409/425/429/5xx、连接中断、返回体非 JSON、`choices` 缺失、模型没给出合法 JSON 全部当作可重试；只有 401/403（密钥问题）立刻放弃
* **自适应限速**——遇到 429 或 5xx 自动拉大请求间隔，恢复后逐步收回；尊重 `Retry-After`
* **可中断**——Ctrl-C 会停止派发并把已完成的写盘，重跑即续

In [ ]:
_lock = threading.Lock()
RETRYABLE = {408, 409, 425, 429, 500, 502, 503, 504, 520, 521, 522, 524, 529}
FATAL     = {401, 403}

class Throttle:
    """全局自适应节流：失败变多就拉开请求间隔，顺利时逐步收回。"""
    def __init__(self):
        self.lk = threading.Lock(); self.delay = 0.0; self.until = 0.0
    def wait(self):
        with self.lk: d, u = self.delay, self.until
        now = time.time()
        if u > now: time.sleep(u - now)
        if d: time.sleep(d * random.uniform(0.5, 1.5))
    def penalise(self, hold=None):
        with self.lk:
            self.delay = min(max(self.delay * 2, 0.3), 8.0)
            if hold: self.until = max(self.until, time.time() + hold)
    def reward(self):
        with self.lk: self.delay *= 0.85
        if self.delay < 0.05: self.delay = 0.0
TH = Throttle()

def extract_json(t):
    t = t.strip()
    for pre in ("```json", "```"):
        if t.startswith(pre): t = t[len(pre):]
    if t.endswith("```"): t = t[:-3]
    t = t.strip()
    try: return json.loads(t)
    except Exception: pass
    s = t.find("{")
    if s < 0: raise ValueError("no JSON in: " + t[:160])
    depth = 0
    for i, ch in enumerate(t[s:], s):
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return json.loads(t[s:i + 1])
    raise ValueError("unbalanced JSON: " + t[:160])

class Truncated(ValueError):
    """输出被 max_tokens 截断——重试无用，需要改配置。"""

def call(prompt):
    """返回 (obj, usage, ms, attempts, err)。err 为 None 表示成功。"""
    body = {"model": MODEL,
            "messages": [{"role": "system", "content": SYS}, {"role": "user", "content": prompt}]}
    if SEND_MAX_TOKENS: body["max_tokens"] = MAX_TOKENS
    if SEND_TEMPERATURE: body["temperature"] = TEMPERATURE
    if JSON_MODE: body["response_format"] = {"type": "json_object"}
    body.update(EXTRA_BODY)
    data = json.dumps(body).encode()
    soft = 0; parse = 0; err = "not attempted"
    for i in range(RETRIES):
        TH.wait(); t0 = time.time()
        try:
            req = urllib.request.Request(BASE_URL, data=data, headers={
                "Content-Type": "application/json", "Authorization": "Bearer " + KEY,
                "Accept": "application/json", "Connection": "close"})
            raw = urllib.request.urlopen(req, timeout=TIMEOUT).read()
            try:
                r = json.loads(raw)
            except Exception:
                raise ValueError("body is not JSON: " + raw[:160].decode("utf-8", "replace"))
            if not isinstance(r, dict) or not r.get("choices"):      # 中转常见：200 但返回错误体
                raise ValueError("no choices: " + json.dumps(r)[:200])
            ch = r["choices"][0]
            fin = ch.get("finish_reason") or ch.get("native_finish_reason")
            msg = (ch.get("message") or {}).get("content")
            if not msg: raise ValueError(f"empty content (finish_reason={fin})")
            try:
                o = extract_json(msg)
            except ValueError as pe:
                if fin == "length" or str(pe).startswith("unbalanced"):
                    raise Truncated(f"output truncated (finish_reason={fin}, usage={r.get('usage')}): {msg[:90]}")
                raise
            if not (isinstance(o.get("risk_24m"), int) and 0 <= o["risk_24m"] <= 100):
                raise ValueError("bad risk_24m: " + str(o)[:120])
            TH.reward()
            return o, r.get("usage", {}), int((time.time() - t0) * 1000), i + 1, None
        except urllib.error.HTTPError as e:
            try: detail = e.read().decode("utf-8", "replace")[:200]
            except Exception: detail = ""
            err = f"HTTP {e.code}: {detail}"
            if e.code in FATAL:
                return None, {}, int((time.time() - t0) * 1000), i + 1, err
            if e.code in RETRYABLE:
                ra = e.headers.get("Retry-After") if e.headers else None
                TH.penalise(float(ra) if (ra or "").replace(".", "").isdigit() else None)
            else:                                     # 400/404 等：中转偶发，给几次机会
                soft += 1
                if soft > SOFT_FATAL_RETRIES:
                    return None, {}, int((time.time() - t0) * 1000), i + 1, err
        except (urllib.error.URLError, socket.timeout, TimeoutError, ConnectionError, OSError) as e:
            err = f"{type(e).__name__}: {str(e)[:160]}"; TH.penalise()
        except Truncated as e:
            err = f"Truncated: {str(e)[:200]}"; parse += 1
            if parse > PARSE_RETRIES:
                return None, {}, int((time.time() - t0) * 1000), i + 1, err
        except ValueError as e:                      # 模型没吐出合法 JSON：偶发，少试几次
            err = f"ValueError: {str(e)[:160]}"; parse += 1
            if parse > PARSE_RETRIES:
                return None, {}, int((time.time() - t0) * 1000), i + 1, err
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:160]}"
        if i < RETRIES - 1:
            time.sleep(min(1.5 * (2 ** i), 45) * random.uniform(0.6, 1.4))
    return None, {}, 0, RETRIES, err

def load_done(path, keys):
    """返回已成功完成的任务键集合；坏行直接跳过。"""
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    r = json.loads(line)
                    if r.get("risk") is not None: done.add(tuple(r[k] for k in keys))
                except Exception:
                    continue
    return done

def run(tasks, out_path, make_prompt, rec_fn, keys, key_of, tag):
    """多轮补跑，直到全部完成或轮次用尽。可随时 Ctrl-C 中断，重跑即续。"""
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    total = len(tasks); t_start = time.time(); stop = threading.Event()
    for p in range(1, PASSES + 1):
        done = load_done(out_path, keys)
        todo = [t for t in tasks if key_of(t) not in done]
        print(f"[{tag}] 第 {p}/{PASSES} 轮：总 {total}，已完成 {len(done)}，待跑 {len(todo)}", flush=True)
        if not todo:
            print(f"[{tag}] 全部完成，用时 {(time.time()-t_start)/60:.1f} 分钟"); return
        fh = open(out_path, "a", encoding="utf-8")
        ok = [0]; bad = [0]; errs = []
        def work(t):
            if stop.is_set(): return
            try:
                o, u, ms, tries, err = call(make_prompt(t))
            except Exception as e:                       # 兜底：绝不让异常冒出线程池
                o, u, ms, tries, err = None, {}, 0, 0, f"{type(e).__name__}: {str(e)[:160]}"
            rec = rec_fn(t)
            rec.update({"ts": time.strftime("%Y-%m-%dT%H:%M:%S"), "model": MODEL,
                        "prompt_sha1": hashlib.sha1(make_prompt(t).encode()).hexdigest(),
                        "risk": (o or {}).get("risk_24m"), "traj": (o or {}).get("trajectory"),
                        "conf": (o or {}).get("confidence"), "evidence": (o or {}).get("evidence"),
                        "would_change": (o or {}).get("would_change_my_mind"),
                        "usage": u, "latency_ms": ms, "attempts": tries, "err": err, "pass": p})
            line = json.dumps(rec, ensure_ascii=False)
            with _lock:
                fh.write(line + "\n"); fh.flush()
                if err: bad[0] += 1; errs.append(err)
                else: ok[0] += 1
                n = ok[0] + bad[0]
                if n % 100 == 0 or n == len(todo):
                    el = time.time() - t_start
                    rate = n / max(el, 1e-9)
                    left = (len(todo) - n) / max(rate, 1e-9) / 60
                    print(f"   {n}/{len(todo)}  成功 {ok[0]}  失败 {bad[0]}  "
                          f"{rate:.1f}/s  预计剩余 {left:.0f} 分钟  节流 {TH.delay:.2f}s", flush=True)
        try:
            with ThreadPoolExecutor(max_workers=WORKERS) as ex:
                futs = [ex.submit(work, t) for t in todo]
                for f in as_completed(futs):
                    try: f.result()
                    except Exception as e: print("   worker 异常(已忽略):", type(e).__name__, str(e)[:90])
        except KeyboardInterrupt:
            stop.set(); print(f"[{tag}] 已中断，进度保存在 {out_path}，重跑本格即续。")
            fh.close(); return
        finally:
            fh.close()
        if bad[0]:
            from collections import Counter
            top = Counter(e.split(":")[0] for e in errs).most_common(3)
            print(f"   本轮失败 {bad[0]}，主要原因: {top}")
            print(f"   示例: {errs[0][:160]}")
        if ok[0] == 0 and bad[0] > 0:
            if any(e.startswith("Truncated") for e in errs):
                print("   ⚠ 输出被截断：把 SEND_MAX_TOKENS 设为 False，或把 MAX_TOKENS 调到 4000 以上")
                print("     （思考型模型的思考 token 也占 max_tokens 额度）")
            else:
                print("   ⚠ 本轮无一成功，请检查密钥、模型名与 BASE_URL。")
            return
        if p < PASSES:
            left = len([t for t in tasks if key_of(t) not in load_done(out_path, keys)])
            if left == 0: break
            print(f"   仍缺 {left}，{PASS_PAUSE}s 后进入下一轮…"); time.sleep(PASS_PAUSE)
    left = len([t for t in tasks if key_of(t) not in load_done(out_path, keys)])
    print(f"[{tag}] 结束：仍缺 {left}（重跑本格可继续补），用时 {(time.time()-t_start)/60:.1f} 分钟")

def report(path, tasks, keys, key_of, tag):
    done = load_done(path, keys); miss = [t for t in tasks if key_of(t) not in done]
    print(f"{tag}: 计划 {len(tasks)}，完成 {len(done)}，缺 {len(miss)}  ({len(done)/len(tasks)*100:.1f}%)")
    return miss

print("执行器就绪")

## 5. 冒烟：3 次调用

**不通就不要往下跑。** 失败时下面会打印完整错误原文。

In [ ]:
assert KEY, f"请先设置环境变量 {KEY_ENV}"

# --- 先做一次裸调用，打印原始返回，确认参数是否合适 ---
_body = {"model": MODEL, "messages": [{"role": "system", "content": SYS},
                                      {"role": "user", "content": build_A(cards[RIDS[0]]["card0"])}]}
if SEND_TEMPERATURE: _body["temperature"] = TEMPERATURE
if JSON_MODE: _body["response_format"] = {"type": "json_object"}
if SEND_MAX_TOKENS: _body["max_tokens"] = MAX_TOKENS
_body.update(EXTRA_BODY)
_req = urllib.request.Request(BASE_URL, data=json.dumps(_body).encode(),
                              headers={"Content-Type": "application/json",
                                       "Authorization": "Bearer " + KEY, "Accept": "application/json"})
_t = time.time()
try:
    _r = json.loads(urllib.request.urlopen(_req, timeout=TIMEOUT).read())
    _ch = _r["choices"][0]; _msg = (_ch.get("message") or {}).get("content") or ""
    print(f"耗时 {time.time()-_t:.1f}s | finish_reason = {_ch.get('finish_reason')}")
    print("usage =", _r.get("usage"))
    print("内容长度", len(_msg), "字符；前 200 字：")
    print(" ", _msg[:200].replace("\n", " "))
    if _ch.get("finish_reason") == "length":
        print("\n⚠ 输出被截断。把 SEND_MAX_TOKENS 设为 False（推荐），或把 MAX_TOKENS 调大。")
except Exception as _e:
    print("裸调用失败:", type(_e).__name__, str(_e)[:200])

# --- 再跑 3 次正式冒烟 ---
print()
for i_, rid in enumerate(RIDS[:3]):
    o, u, ms, tries, err = call(build_A(cards[rid]["card0"]))
    print(f"[{i_+1}] err={err}  tries={tries}  {ms}ms  risk={(o or {}).get('risk_24m')}  usage={u}")

## 6. 阶段 A（900 次）

In [ ]:
tasksA = [(r, k) for r in RIDS for k in range(REPEATS_A)]
random.Random(SEED).shuffle(tasksA)
run(tasksA, OUT_A, lambda t: build_A(cards[t[0]]["card0"]),
    lambda t: {"stage": "A_lowinfo", "RID": t[0], "rep": t[1],
               "dx24": cards[t[0]].get("dx24"), "dx36": cards[t[0]].get("dx36")},
    ["RID", "rep"], lambda t: (t[0], t[1]), "阶段A")
_ = report(OUT_A, tasksA, ["RID", "rep"], lambda t: (t[0], t[1]), "阶段A")

## 7. 阶段 A 结果与阶段 B 名单

先用全员跑 `B_independent` 测出真实落差，再按落差挑人。

In [ ]:
dA = pd.DataFrame([json.loads(l) for l in open(OUT_A, encoding="utf-8")])
dA = dA[dA.risk.notna()]
gA = dA.groupby("RID").agg(riskA=("risk", "mean"), sdA=("risk", "std"),
                           dx24=("dx24", "first"), dx36=("dx36", "first")).reset_index()
gA["y24"] = np.where(gA.dx24.isin(["CN", "MCI", "Dementia"]), (gA.dx24 == "Dementia").astype(float), np.nan)
ev = gA.dropna(subset=["y24"]).copy()
ev["wrong"] = ((ev.riskA >= 50).astype(int) != ev.y24).astype(int)
print(f"可评估 {len(ev)}  A判错 {int(ev.wrong.sum())} ({ev.wrong.mean()*100:.1f}%)  运行间SD中位 {ev.sdA.median():.2f}")

AT_ALL = {}
for rid in ev.RID:
    s = dA[dA.RID == rid]
    mid = s.iloc[(s.risk - s.risk.mean()).abs().values.argmin()]
    AT_ALL[rid] = (f"Risk of progression to dementia within 24 months: {int(mid.risk)}%. "
                   f"Trajectory: {mid.traj}. Reasoning: " + "; ".join(mid.evidence or []))
INFO_ALL = ev.set_index("RID")[["riskA", "wrong", "dx24", "dx36"]].to_dict("index")
ALL_IDS = sorted(AT_ALL)

print("\n【预注册预测】")
print("  1. 落差>35 组的 B_serial 牵引系数 ≥ 0.12")
print("  2. 落差>35 组中 B_serial 的错误滞留率高于 B_independent")
print("  3. B_irrelevant 均值平移≈0，提及率<1%")
print("  无论结果是否符合，全部条件一律照实报告。\n")

AT, INFO = AT_ALL, INFO_ALL
KEYB = ["RID", "cond", "rep"]; KOFB = lambda t: (t[0], t[1], t[2])
recB = lambda t: {"stage": "B", "RID": t[0], "cond": t[1], "rep": t[2],
                  "riskA": INFO[t[0]]["riskA"], "wrongA": INFO[t[0]]["wrong"],
                  "dx24": cards[t[0]].get("dx24"), "dx36": cards[t[0]].get("dx36")}
mkB = lambda t: build_B(cards[t[0]]["card12"], AT[t[0]], CONDS_B[t[1]])

tasks0 = [(r, "B_independent", k) for r in ALL_IDS for k in range(REPEATS_B)]
random.Random(SEED).shuffle(tasks0)
run(tasks0, OUT_B, mkB, recB, KEYB, KOFB, "筛选轮")

### 按落差挑人

In [ ]:
N_PER_ARM = 75          # 每组（A判错/A判对）取落差最大的多少人
d0 = pd.DataFrame([json.loads(l) for l in open(OUT_B, encoding="utf-8")])
d0 = d0[(d0.cond == "B_independent") & d0.risk.notna()]
S = d0.groupby("RID").risk.mean().rename("indep").reset_index()
S["riskA"] = S.RID.map(lambda r: INFO_ALL[r]["riskA"])
S["wrong"] = S.RID.map(lambda r: INFO_ALL[r]["wrong"])
S["absgap"] = (S.riskA - S.indep).abs()
print(f"全员落差: 中位 {S.absgap.median():.1f}  75分位 {S.absgap.quantile(.75):.1f}  最大 {S.absgap.max():.1f}"
      f"  |  >35 的有 {(S.absgap>35).sum()} 人")
pick = pd.concat([g.nlargest(N_PER_ARM, "absgap") for _, g in S.groupby("wrong")])
SEL = sorted(pick.RID.tolist())
AT   = {r: AT_ALL[r] for r in SEL}
INFO = {r: INFO_ALL[r] for r in SEL}
print(f"入选 {len(SEL)} 人（A判错 {int((pick.wrong==1).sum())} / A判对 {int((pick.wrong==0).sum())}）"
      f"  落差中位 {pick.absgap.median():.1f}")

## 8. 阶段 B（其余 4 个条件）

In [ ]:
tasksB = [(r, c, k) for r in SEL for c in CONDS_B for k in range(REPEATS_B)]
random.Random(SEED).shuffle(tasksB)
run(tasksB, OUT_B, mkB, recB, KEYB, KOFB, "阶段B")
missB = report(OUT_B, tasksB, KEYB, KOFB, "阶段B")

## 9. 补缺（可反复运行）

如果上一格结束后仍有缺口，**直接重复运行这一格**即可，每次只跑还缺的部分。
中转彻底不通时，先等一会儿再跑。

In [ ]:
missB = report(OUT_B, tasksB, KEYB, KOFB, "阶段B")
if missB:
    print("开始补缺…")
    run(missB, OUT_B, mkB, recB, KEYB, KOFB, "补缺")
    report(OUT_B, tasksB, KEYB, KOFB, "阶段B")
else:
    print("无缺口，可以进入分析。")

bad = pd.DataFrame([json.loads(l) for l in open(OUT_B, encoding="utf-8")])
bad = bad[bad.err.notna()]
if len(bad):
    print("\n累计失败记录", len(bad), "条（不影响分析，成功记录会覆盖同一任务）")
    print(bad.err.astype(str).str.slice(0, 60).value_counts().head(5).to_string())

## 10. 分析

In [ ]:
dB = pd.DataFrame([json.loads(l) for l in open(OUT_B, encoding="utf-8")])
dB = dB[dB.risk.notna()]
full = dB.groupby("RID").cond.nunique()
COMPLETE = sorted(full[full == len(CONDS_B)].index)
print(f"文件内病人 {dB.RID.nunique()}，条件跑齐的 {len(COMPLETE)}，仅有筛选轮记录的 {dB.RID.nunique()-len(COMPLETE)} 人已排除")
dB = dB[dB.RID.isin(COMPLETE) & dB.RID.isin(SEL)]
W = dB.pivot_table(index="RID", columns="cond", values="risk", aggfunc="mean")
W["riskA"] = [INFO_ALL[r]["riskA"] for r in W.index]
W["wrongA"] = [INFO_ALL[r]["wrong"] for r in W.index]
W["dx36"] = [cards[r].get("dx36") for r in W.index]
W["y36"] = np.where(pd.Series(W.dx36.values).isin(["CN","MCI","Dementia"]).values,
                    (W.dx36=="Dementia").astype(float), np.nan)
W["gap"] = W.riskA - W.B_independent
sdB = dB.groupby(["cond","RID"]).risk.std().groupby("cond").mean()

print("="*64)
print(f"病人 {len(W)}   有36月结局 {int(W.y36.notna().sum())}")
print(f"落差 |gap| 中位 {W.gap.abs().median():.1f}  75分位 {W.gap.abs().quantile(.75):.1f}  最大 {W.gap.abs().max():.1f}")
print(f"阶段B 同条件跨重复平均SD {sdB.mean():.2f}（接近0 则下面的变动都不是采样噪声）")

print("\n【1】相对 B_independent 的平移")
rows=[]
for c in [k for k in CONDS_B if k!="B_independent"]:
    x=(W[c]-W.B_independent).dropna()
    rows.append([c, x.mean(), 1.96*x.sem(), x.abs().mean(),
                 stats.wilcoxon(x).pvalue if x.abs().sum() else np.nan])
print(pd.DataFrame(rows, columns=["条件","均值平移","±95%CI","平均绝对变动","p"]).round(3).to_string(index=False))
print("  ↑ B_irrelevant 是阴性对照：它的均值平移应≈0；它的绝对变动代表『换任何上下文都会有的抖动』")

print("\n【2】牵引系数（每 1 分落差被拉走多少）")
def slope(s, col):
    m = s.dropna(subset=["gap", col, "B_independent"])
    X = np.column_stack([np.ones(len(m)), m.gap.values]); y = (m[col]-m.B_independent).values
    b = np.linalg.lstsq(X, y, rcond=None)[0]; e = y-X@b
    se = np.sqrt(np.diag(e@e/max(len(m)-2,1)*np.linalg.inv(X.T@X)))
    return b[1], 1.96*se[1], len(m)
for c in [k for k in CONDS_B if k!="B_independent"]:
    k_, ci, n = slope(W, c)
    print(f"  {c:15s} {k_:+.3f} ±{ci:.3f}  (n={n})")
print("  参照：注入实验 +0.098 [0.083,0.113]；前一轮高信息流水线 +0.098 [0.034,0.161]")

print("\n【3】按落差分层（仅 B_serial，|gap|≥10 才可靠）")
W["bin"]=pd.cut(W.gap.abs(), [0,10,20,35,200], labels=["<10","10-20","20-35",">35"])
for b_, s in W.groupby("bin", observed=True):
    if len(s)>15:
        k_, ci, n = slope(s, "B_serial")
        print(f"  |gap| {b_:6s} n={n:3d}  牵引系数 {k_:+.3f} ±{ci:.3f}  平均绝对平移 {(s.B_serial-s.B_independent).abs().mean():.2f}")

print("\n【4】2x2 净收益（仅有36月结局者，阈值50）")
ev2 = W.dropna(subset=["y36"]).copy()
acc = lambda c: ((ev2[c]>=50).astype(int)==ev2.y36)
T=[]
for c in CONDS_B:
    ok=acc(c)
    T.append({"条件":c,
              "A判错组正确":int(ok[ev2.wrongA==1].sum()), "A判错组n":int((ev2.wrongA==1).sum()),
              "A判对组正确":int(ok[ev2.wrongA==0].sum()), "A判对组n":int((ev2.wrongA==0).sum()),
              "错误滞留率%":round((~ok[ev2.wrongA==1]).mean()*100,1)})
T=pd.DataFrame(T)
T["A判错组%"]=(T.A判错组正确/T.A判错组n*100).round(1); T["A判对组%"]=(T.A判对组正确/T.A判对组n*100).round(1)
print(T[["条件","A判错组%","A判对组%","错误滞留率%"]].to_string(index=False))
b=T.set_index("条件")
print("\n  相对 B_serial 的净收益：")
for c in [k for k in CONDS_B if k!="B_serial"]:
    save=int(b.loc[c,"A判错组正确"]-b.loc["B_serial","A判错组正确"])
    hurt=int(b.loc[c,"A判对组正确"]-b.loc["B_serial","A判对组正确"])
    print(f"    {c:15s} A判错时多救 {save:+d}   A判对时多错 {-hurt:+d}   净 {save+hurt:+d}")

print("\n【5】B 会不会承认看了上游判断")
PRIOR=r"previous|prior assessment|baseline assessment|earlier assessment|upstream|prior opinion"
hasp=lambda L: any(re.search(PRIOR,s.lower()) for s in L) if isinstance(L,list) else False
for c in CONDS_B:
    s=dB[dB.cond==c]
    sh=(W[c]-W.B_independent).mean() if c!="B_independent" else 0.0
    print(f"  {c:15s} 平移{sh:+6.2f}   提及 {s.evidence.apply(hasp).mean()*100:5.1f}%")

print("\n【6】方向性")
for c in [k for k in CONDS_B if k!="B_independent"]:
    hi=W[W.gap>0]; lo=W[W.gap<0]
    print(f"  {c:15s} A偏高时 {(hi[c]-hi.B_independent).mean():+6.2f} (n={len(hi)})   "
          f"A偏低时 {(lo[c]-lo.B_independent).mean():+6.2f} (n={len(lo)})")
print("="*64)

## 11. 结论速查

| 看什么 | 判读 |
|---|---|
| `B_irrelevant` 均值平移 | ≈0 → 实验没跑偏 |
| `B_irrelevant` 平均绝对变动 | 明显小于其他条件 → 抖动是 A 的判断特有 |
| `B_serial` 牵引系数 | 与注入实验反推值接近 → 机制可迁移 |
| `B_sourced` / `B_firewall` 系数 | 明显低于 `B_serial` → 缓解有效 |
| `\|gap\|>35` 一行 | 系数仍≈0.1 或更高 → 上游越离谱危害越大 |
| 提及率 | <1% → 审阅理由无法发现污染 |

